In [7]:
# import both datasets with significant features and target value 

from pathlib import Path # access to working directory 
import pandas as pd # data manipulation 

# current working directory 
dir = Path.cwd()

# look into Data and find linear.csv and non_linear.csv
data_path = dir.parent / 'Data' / 'linear.csv'
data_path2 = dir.parent / 'Data' / 'non_linear.csv'
target_path = dir.parent / 'Data' / 'target.csv'

# load data path into csv
linear = pd.read_csv(data_path)
non_linear = pd.read_csv(data_path2)
Y = pd.read_csv(target_path)


In [8]:
# training phase for simpler models
# import sklearn train test split 
from sklearn.model_selection import train_test_split

# target value
Y = Y['Survived']

# covariate variables (linear)
X = linear[['Pclass', 'Age', 'SibSp', 'Sex_female', 'Cabin_encoded']]

# 60/40 train/test split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state = 42)


In [9]:
# Modeling Phase (Classification)
# import sklearn numeric performance indicators for downstream modeling workflow 
from sklearn.metrics import confusion_matrix, classification_report 


## logistic Regression
Performance Metrics: 
- Recall: 81% of positve cases were caught by the model, signaling a reduction in false negatives 
- Precision: 75% of positive cases were accurately predicted, reducing the amount of false positives
- Fl Score: The harmonic mean was calcuated ay 78%, signaling a level of usefulness for the model




In [10]:
# basline model (logistic regression) 

from sklearn.linear_model import LogisticRegression

# begin instantiantion process
lr = LogisticRegression(max_iter= 100)

# fit data on model
lr.fit(X_train, Y_train)

# predictions 
lr_pred = lr.predict(X_test)

# performance metrrics
print(f'Classification Report {classification_report(Y_test, lr_pred)}')
print(f'Confusion matric {confusion_matrix(Y_test, lr_pred)}')


Classification Report               precision    recall  f1-score   support

           0       0.84      0.87      0.86       157
           1       0.81      0.77      0.79       111

    accuracy                           0.83       268
   macro avg       0.83      0.82      0.82       268
weighted avg       0.83      0.83      0.83       268

Confusion matric [[137  20]
 [ 26  85]]


In [11]:
# Let's find the best hyperparameter settings for the model 
from sklearn.model_selection import GridSearchCV

# parameters
params = [ 
    {
        'solver': ['liblinear'], 
        'penalty': ['l1', 'l2'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 50, 100],
        'max_iter': [2000],
        'fit_intercept': [True], 
    
    }, 
    { 
        'solver': ['lbfgs'], 
        'penalty': ['l2'], # lbfgs cannot do L1
        'C': [0.001, 0.1, 1, 10, 100],
        'max_iter': [2000], 
        'fit_intercept': [True]

    }, 
    {
        'solver': ['saga'], 
        'penalty': ['elasticnet'],
        'C': [0.01, 0.1, 1, 10, 100], 
        'l1_ratio': [0.5], # required for elasticnet
        'max_iter': [2000], 
        'fit_intercept': [True]
    }
]


# instantiantion process 
lr_best = GridSearchCV(estimator= lr, param_grid= params, cv = 5, verbose= 5, n_jobs= -1)

# fit data on model
lr_best.fit(X_train, Y_train)

# predictions
lr_pred = lr_best.predict(X_test)

# performance metrics 
print(f'Classification Report C{classification_report(Y_test, lr_pred)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, lr_pred)}')
print(f'Best settings {lr_best.best_estimator_}')

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Classification Report C              precision    recall  f1-score   support

           0       0.83      0.88      0.85       157
           1       0.81      0.75      0.78       111

    accuracy                           0.82       268
   macro avg       0.82      0.81      0.82       268
weighted avg       0.82      0.82      0.82       268

Confusion Matrix [[138  19]
 [ 28  83]]
Best settings LogisticRegression(C=10, max_iter=2000, penalty='l1', solver='liblinear')


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


## Support Vector Machines

Peformance Metrics:
- Recall: 70% of postives cases were caught by the model, reducing some level false negatives. 
- precision: 80% of positive cases were accurately predicted by the model, reduing false positves.
- The harmonic mean (f1) outputted a 75%, suggesting the model has a moderate use case in predicting  

Key Insights: 
- In the process of utilizing GridSeacrchCv, the model had trouble converging. High numerical SVM parameters work extremely hard to achieve 100% accuracy on the training data. Thus, giving the model a hard time to come up with a prediction within a certiain timeframe even with the implementation of n_jobs (a parameters that tells the model to use all cpu's inside of a computer). The optimal solution in this case was to envoke HavlingGridSearchCV (a derivitive of GridSearchCV with the ability to use a subset of data points, leading to faster predictions and best setting outputs)

In [12]:
from sklearn.svm import SVC # support vector model 
# instantiation process
svm = SVC()

# fit the data on the model
svm.fit(X_train, Y_train)

# predictios
Y_predictions = svm.predict(X_test)

# predictions results
Y_predictions

# Evaluate model performance results 
print(f'Classification Report {classification_report(Y_test, Y_predictions)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, Y_predictions)}')

Classification Report               precision    recall  f1-score   support

           0       0.80      0.92      0.86       157
           1       0.86      0.68      0.76       111

    accuracy                           0.82       268
   macro avg       0.83      0.80      0.81       268
weighted avg       0.83      0.82      0.82       268

Confusion Matrix [[145  12]
 [ 36  75]]


In [ ]:
# find the best settings for the SVM model 

# parameters
params = [ 
    {
        'kernal': ['linear'], 
        'C': [0.01, 0.1, 10, 100], 
        'gamma': [0]

    }
]

# Grid Search settings 
svm_best = GridSearchCV(estimator = svm, param_grid = params, cv =5, n_jobs = -1, verbose = 5)

# fit data onto model
svm_best.fit(X_train, Y_train)

# predictions
svm_pred = svm_best.predict(X_test)

# performance result 
print(f'Classification Report: {classification_report(Y_test, svm_pred)}')
print(f'confusion Matrix: {confusion_matrix(Y_test, svm_pred)}')
print(f'Best Settings: {svm_best.best_estimator_}')

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Classification Report:               precision    recall  f1-score   support

           0       0.80      0.92      0.85       157
           1       0.85      0.67      0.75       111

    accuracy                           0.81       268
   macro avg       0.82      0.79      0.80       268
weighted avg       0.82      0.81      0.81       268

confusion Matrix: [[144  13]
 [ 37  74]]
Best Settings: SVC(C=10, gamma=0.1)


## K-NearestNeighbors Algorithm

Metrics: 
- Recall: 73% of the positive cases we're identified and caught by the model, reducing false negatives
- Precision: 79% of positive cases we're accurately predicted by the model, reducing false posiitives 
- F1 Score: The harmonic mean of the two metric are 81% signaling a usefulness in the model predictions for real world implementation

Key Insights: Model peforms 5% better when it focuses on capturing the best recall score! 


In [ ]:
# (KNN) uses distances to classify which class a data point belongs in 

from sklearn.neighbors import KNeighborsClassifier 

# instantiation process
KNN = KNeighborsClassifier(n_neighbors=5)

# fit the data on the model 
KNN.fit(X_train, Y_train)

# predictions
KNN_pred = KNN.predict(X_test)

# performance results 
print(f'Classification Report: {classification_report(Y_test, KNN_pred)}')
print(f'Confusion Matric: {confusion_matrix(Y_test, KNN_pred)}')

In [ ]:
# Find the best optimal settings for K-Nearest Neighbors 
# import gridsearchcv to locate the best paramaters for the model
from sklearn.model_selection import GridSearchCV

params = {'n_neighbors': [1, 5, 10, 15, 20], 
          'weights': ['uniform', 'distance'], 
          'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 
          'leaf_size': [10, 20, 30, 40], }
# instantiation process
KNN_best = GridSearchCV(estimator = KNN, param_grid= params, cv = 5, verbose = 5, scoring = 'recall')

# fit data on the model 
KNN_best.fit(X_train, Y_train)

# predictions 
KNN_pred = KNN_best.predict(X_test)

# performance results
print(f'Classification Report:{classification_report(Y_test, KNN_pred)}')
print(f'Best Settings: {(KNN_best.best_estimator_)}')

## Random Forest Classification
- Performance Metrics: 
- Recall: 67% of positive cases were caught and predicted by the model, which entails the model is weak/useless in predicting 
- Precision 83% of positive cases were accurately predicted by the model, reducing the amouf of false positives. 
- F1 Score: The harmonic mean calculated between the recall and precision adjusted for 73%, signaling a slight usefulness of the model.  

In [ ]:
# Random Forest Classification 
from sklearn.ensemble import RandomForestClassifier 

# instantiation process
rf = RandomForestClassifier(n_estimators= 100)

# fit data onto model 
rf.fit(X_train, Y_train)

# predictions
rf_pred = rf.predict(X_test)

# peformance evaluation
print(f'Classification Repor: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')

In [ ]:
# Find best parameters for the model 
# params 
params = {'n_estimators': [100, 200, 300, 400, 500,], 
          'max_depth': [None], 
          'min_samples_split': [2, 5, 10, 0.05],
          'min_samples_leaf': [1, 2, 4, 0.5],
          'max_features': ['sqrt', 'log2'],
          'random_state': [42], 
          'n_jobs': [-1]}

# instantation process 
rf_best = GridSearchCV(estimator= rf, param_grid= params, cv = 5, verbose = 5, n_jobs = -1, error_score= 'raise')
# fit data onto model 
rf_best.fit(X_train, Y_train)

# predicitions
rf_pred = rf_best.predict(X_test)

# performance summary 
print(f'Classification Report: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')
print(f'Best Settings: {rf_best.best_estimator_}')

# XGboost Algorithm 
Performance Metrics: 
- Recall: 70% of cases were caught by the model, reducing false negative in process.
- Precision: 80% of cases were accurately predicted by the model, which led to the reduction in false positives. 
- F1: The harmonic mean of recall and precision adjusted for 75% signaling a moderate usefullness in real world application

In [ ]:
# Let's use a boosted algorithm to capture omore accuacry and performance 

# import package 
import xgboost as xgb 

# intialize xgboostclassifier
xgb = xgb.XGBClassifier(n_estimators = 100)
# fit data on boosted algo
xgb.fit(X_train, Y_train)

# predictions
xgb_pred = xgb.predict(X_test)

# performance metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusin Matrix {confusion_matrix(Y_test, xgb_pred)}')



In [ ]:
# Let's find the best settings for the boosted algorithm

# parameters 
params = {'n_estimators': [100, 200, 500, 1000], 
          'learning_rate': [0.1, 0.2, 5, 10], 
          'max_depth': [5, 10, 20, 30, 40]}

# intitalize gridsearch 
xgb_best = GridSearchCV(estimator= xgb, param_grid= params, cv = 5, verbose = 5, n_jobs =-1) 

# fit data on model
xgb_best.fit(X_train, Y_train)

# predcitions
xgb_pred = xgb.predict(X_test)

# evaluation metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusion Matirx {confusion_matrix(Y_test, xgb_pred)}')
print(f'Best settings {xgb_best.best_estimator_}')
